# Which Berke lab sessions need sorting / decoding?

Finds every Berke lab hex maze session and works out what the **full sorting + decode
pipeline** (`Berke_Lab_Sorting_and_Decode_V1.ipynb`) still needs to be run on -- the same
detect-what-can-be-populated idea as `Hex_Maze_Theta.ipynb` and `Populate_All_Decoding.ipynb`.

That pipeline is: sort groups -> SpikeSortingRecording -> ArtifactDetection -> SpikeSorting
-> CurationV1 -> MetricCuration -> SpikeSortingOutput -> SortedSpikesGroup + PositionGroup
-> SortedSpikesDecodingV1 -> DecodingOutput.

So a session is *runnable* when it has:
- **raw ephys** (`Raw`) -- otherwise there is nothing to sort
- **sort groups** (`spikesorting.v1.SortGroup`) -- the pipeline sorts per sort group
- **Trodes position** (`PositionOutput.TrodesPosV1`) -- what `PositionGroup` is built from,
  needed for the decode half

This notebook only reads -- it doesn't populate anything.

## Find the Berke lab hex maze sessions

In [ ]:
import pandas as pd

import spyglass.common as sgc
import spyglass.spikesorting.v1 as sgs
from spyglass.common import Raw
from spyglass.position import PositionOutput
from spyglass.decoding.decoding_merge import DecodingOutput
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
from spyglass_hexmaze.hex_maze_behavior import HexMazeBlock

# All hex maze sessions, with their lab + subject
hex_sessions = sorted(set(HexMazeBlock.fetch('nwb_file_name')))
session_keys = [{'nwb_file_name': s} for s in hex_sessions]
session_lab = pd.DataFrame(
    (sgc.Session & session_keys).fetch('nwb_file_name', 'subject_id', 'lab_name', as_dict=True)
)

# Just the Berke lab ones (these are the IM-* sessions, sorted with spikesorting v1)
berke = session_lab[session_lab['lab_name'] == 'Berke Lab'].copy()
berke_sessions = sorted(berke['nwb_file_name'])

print(f"{len(berke_sessions)} Berke lab hex maze sessions "
      f"across {berke['subject_id'].nunique()} subjects")

## Check each pipeline prerequisite

Each of these is session-level for Berke (one run epoch per session, `00_r1`).

In [ ]:
berke_keys = [{'nwb_file_name': s} for s in berke_sessions]

# --- Raw ephys: nothing to sort without it ---
ephys_sessions = set((Raw & berke_keys).fetch('nwb_file_name'))

# --- Sort groups: Berke IM-* sessions live in spikesorting v1 ---
# (the pipeline sorts per sort group, so these must exist first -- set_group_by_shank)
sort_group_sessions = {
    s for s in berke_sessions if len(sgs.SortGroup & {'nwb_file_name': s}) > 0
}

# --- Trodes position: what PositionGroup is built from for the decode half ---
trodes = pd.DataFrame(
    PositionOutput.TrodesPosV1.fetch('nwb_file_name', 'interval_list_name', as_dict=True)
)
trodes_sessions = set(trodes['nwb_file_name']) & set(berke_sessions) if len(trodes) else set()

# --- Already sorted? (walk back through the v1 recording selection) ---
# NOTE: don't use merge_fetch('nwb_file_name') -- the v1 part is keyed by sorting_id, so it
# would be skipped and every v1-sorted session would look unsorted.
sorted_sessions = {
    s for s in berke_sessions
    if len(SpikeSortingOutput().get_restricted_merge_ids(
        {'nwb_file_name': s}, sources=['v0', 'v1'], restrict_by_artifact=False))
}

# --- Already decoded? (walk the DecodingOutput parts) ---
dec_frames = []
for part in DecodingOutput().parts(as_objects=True):
    if 'nwb_file_name' in part.heading.names:
        dec_frames.append(pd.DataFrame(part.fetch('nwb_file_name', as_dict=True)))
dec = pd.concat(dec_frames, ignore_index=True) if dec_frames else pd.DataFrame(columns=['nwb_file_name'])
decoded_sessions = set(dec['nwb_file_name']) & set(berke_sessions)

print(f"raw ephys:      {len(ephys_sessions)} / {len(berke_sessions)}")
print(f"sort groups:    {len(sort_group_sessions)} / {len(berke_sessions)}")
print(f"trodes position:{len(trodes_sessions)} / {len(berke_sessions)}")
print(f"sorted:         {len(sorted_sessions)} / {len(berke_sessions)}")
print(f"decoded:        {len(decoded_sessions)} / {len(berke_sessions)}")

## What does each session need next?

The pipeline is sequential, so we report the **first** missing step for each session.

In [ ]:
berke['has_ephys'] = berke['nwb_file_name'].isin(ephys_sessions)
berke['has_sort_group'] = berke['nwb_file_name'].isin(sort_group_sessions)
berke['has_position'] = berke['nwb_file_name'].isin(trodes_sessions)
berke['has_sorting'] = berke['nwb_file_name'].isin(sorted_sessions)
berke['has_decode'] = berke['nwb_file_name'].isin(decoded_sessions)


def next_step(row):
    """First missing step in the sorting -> decode pipeline for this session."""
    if not row.has_ephys:
        return 'no ephys (cannot sort)'
    if not row.has_sort_group:
        return '1. create sort groups'
    if not row.has_sorting:
        return '2. run sorting'
    if not row.has_position:
        return '3. process position (needed to decode)'
    if not row.has_decode:
        return '4. run decode'
    return 'done'


berke['next_step'] = berke.apply(next_step, axis=1)

candidates = (
    berke[['subject_id', 'nwb_file_name', 'has_ephys', 'has_sort_group',
           'has_position', 'has_sorting', 'has_decode', 'next_step']]
    .sort_values(['next_step', 'subject_id', 'nwb_file_name'])
    .reset_index(drop=True)
)

print('Berke sessions by next step:')
print(candidates['next_step'].value_counts().to_string())
display(candidates)

## The run list

Sessions ready for each stage, as `nwb_file_name`s you can paste straight into
`Berke_Lab_Sorting_and_Decode_V1.ipynb`.

In [ ]:
# Sessions that have everything the sorting pipeline needs and just haven't been run yet
ready_to_sort = candidates.loc[candidates['next_step'] == '2. run sorting', 'nwb_file_name'].tolist()
# Sorted already, position ready, just needs the decode half
ready_to_decode = candidates.loc[candidates['next_step'] == '4. run decode', 'nwb_file_name'].tolist()
# Have ephys but no sort groups -- do this first (set_group_by_shank), then they can be sorted
need_sort_groups = candidates.loc[candidates['next_step'] == '1. create sort groups', 'nwb_file_name'].tolist()
# Sorted but missing position, so the decode half is blocked
need_position = candidates.loc[candidates['next_step'] == '3. process position (needed to decode)', 'nwb_file_name'].tolist()

for label, sessions in [
    ('READY TO SORT (run the full pipeline)', ready_to_sort),
    ('READY TO DECODE (sorted already)', ready_to_decode),
    ('NEED SORT GROUPS FIRST', need_sort_groups),
    ('NEED POSITION FIRST', need_position),
]:
    print(f"\n{label}: {len(sessions)}")
    for s in sessions:
        print(f"    {s!r},")

n_done = int((candidates['next_step'] == 'done').sum())
n_no_ephys = int((candidates['next_step'] == 'no ephys (cannot sort)').sum())
print(f"\n{n_done} session(s) fully done; {n_no_ephys} have no ephys (behavior/photometry only).")